**Purpose:** Build the three business-ready Gold tables from nyc_mobility.silver.trips.

**Owner:** Course Author

**Inputs:** nyc_mobility.silver.trips (Delta table).

**Outputs:** Three Delta tables in nyc_mobility.gold (daily summary, hourly demand, peak hours analysis).

In [0]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/Workspace/Users/adarshkumar7799@gmail.com/nyc-mobility-analytics/src")

from nyc_mobility import bronze, silver, gold, transforms

print("Modules imported successfully.")

In [0]:
from pyspark.sql import functions as F

SILVER_TRIPS_TABLE = "nyc_mobility.silver.trips"
GOLD_REVENUE_BY_DATE_TABLE = "nyc_mobility.gold.revenue_by_date"
GOLD_TRIPS_BY_BOROUGH_TABLE = "nyc_mobility.gold.trips_by_borough"
GOLD_TRIPS_BY_HOUR_WEEKEND_TABLE = "nyc_mobility.gold.trips_by_hour_weekend"

# Gold reads exactly one table: silver. No filters, no joins, no dedup —
# silver already applied every agreed quality rule, and null/unknown
# grouping keys form their own group and are kept.
silver_df = spark.table(SILVER_TRIPS_TABLE)

# 1. Revenue by date: one row per pickup date. Revenue is total_amount,
# cast to DECIMAL(12,2) per row before summing, rounded once at the end.
revenue_by_date_df = (
    silver_df
    .groupBy(F.to_date(F.col("tpep_pickup_datetime")).alias("pickup_date"))
    .agg(
        F.round(
            F.sum(F.col("total_amount").cast("DECIMAL(12,2)")), 2
        ).alias("total_revenue"),
        F.count("*").alias("trip_count"),
    )
    .orderBy(F.col("pickup_date").asc())
)

(
    revenue_by_date_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_REVENUE_BY_DATE_TABLE)
)
print(
    f"Wrote {GOLD_REVENUE_BY_DATE_TABLE}: "
    f"{spark.table(GOLD_REVENUE_BY_DATE_TABLE).count()} rows"
)

# 2. Trips by borough: one row per pickup borough.
trips_by_borough_df = (
    silver_df
    .groupBy(F.col("pickup_borough"))
    .agg(
        F.count("*").alias("trip_count"),
        F.round(
            F.sum(F.col("total_amount").cast("DECIMAL(12,2)")), 2
        ).alias("total_revenue"),
    )
    .orderBy(F.col("trip_count").desc())
)

(
    trips_by_borough_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TRIPS_BY_BOROUGH_TABLE)
)
print(
    f"Wrote {GOLD_TRIPS_BY_BOROUGH_TABLE}: "
    f"{spark.table(GOLD_TRIPS_BY_BOROUGH_TABLE).count()} rows"
)

# 3. Trips by hour and weekend flag: one row per hour and weekend flag.
trips_by_hour_weekend_df = (
    silver_df
    .groupBy(F.col("pickup_hour"), F.col("is_weekend"))
    .agg(F.count("*").alias("trip_count"))
    .orderBy(F.col("pickup_hour").asc(), F.col("is_weekend").asc())
)

(
    trips_by_hour_weekend_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TRIPS_BY_HOUR_WEEKEND_TABLE)
)
print(
    f"Wrote {GOLD_TRIPS_BY_HOUR_WEEKEND_TABLE}: "
    f"{spark.table(GOLD_TRIPS_BY_HOUR_WEEKEND_TABLE).count()} rows"
)